# Step 2: Chunk, embed and index (about 45-60 minutes)

**Where you are:** Step 1 collected and cleaned the pages. Step 2 turns them into something you can **search by meaning**. It finishes the ingestion half of RAG:

Collect → Clean → **Chunk → Embed → Index**

**Three ideas in this step**
1. **Chunking:** split each page into small passages. A page can cover ten topics; a question usually needs one paragraph. Small passages let retrieval return exactly the relevant part, and they fit in the model's limits.
2. **Embedding:** turn each chunk into a vector (a list of numbers) that captures its meaning.
3. **Indexing:** store the vectors in a structure that quickly finds the closest ones to a question's vector.

**This step costs nothing.** It uses a free open-source embedding model that runs inside Colab.

## 2.1 Setup and load the documents

**Concepts**
- `sentence-transformers` loads open-source embedding models from HuggingFace.
- `faiss-cpu` is a library from Meta for fast vector search.
- We read `docs.json` from the Drive folder used in Step 1.

In [1]:
%pip install -q sentence-transformers faiss-cpu
import os, re, json, statistics
import numpy as np

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/docs-rag"
else:
    BASE = "/content/docs-rag"
DATA = f"{BASE}/data"

docs = json.load(open(f"{DATA}/docs.json"))
print(len(docs), "pages loaded |", sum(d["words"] for d in docs.values()), "words")

Mounted at /content/drive
28 pages loaded | 40545 words


**What you should see:** the same page and word counts you noted in Step 1. If the file is not found, check that Step 1 saved to Drive (`USE_DRIVE = True`).

## 2.2 Choosing a chunk size

**Concepts**
- Models read **tokens**, not words. One English word is roughly 1.3 tokens.
- An embedding model has a **maximum input length**. Our model, `BAAI/bge-small-en-v1.5`, reads up to 512 tokens. Anything longer is **silently cut off**, so the end of a long chunk would never be searchable. That is a classic hidden bug.
- **Too small** (a sentence): chunks lose context ("it supports this" — what is "it"?).
- **Too large** (a whole page): one vector has to represent many topics, so it matches nothing well, and you send the LLM lots of irrelevant text.
- A good starting point is **about 300 words (roughly 400 tokens)**. Treat it as a setting to tune with evaluation later (Step 9).

**Overlap:** consecutive chunks share a little text (about 50 words). If an answer sits on a boundary between two chunks, overlap makes sure at least one chunk contains it whole.

In [2]:
from sentence_transformers import SentenceTransformer
emb = SentenceTransformer("BAAI/bge-small-en-v1.5")
print("Max tokens the model reads:", emb.max_seq_length)

def n_tokens(text):
    return len(emb.tokenizer(text)["input_ids"])

sample = next(iter(docs.values()))["text"]
words300 = " ".join(sample.split()[:300])
print("300 words =", n_tokens(words300), "tokens")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Max tokens the model reads: 512
300 words = 398 tokens


**What you should see:** `Max tokens ... 512` and roughly 350-450 tokens for 300 words, safely under the limit.

## 2.3 Structure-aware chunking

**Concepts**
- The simplest method cuts every 300 words regardless of content (**fixed-size chunking**). It can cut a sentence or a table in half.
- In Step 1 we kept **one paragraph per line**. So we can do better: **pack whole paragraphs** into a chunk until it reaches about 300 words, then start a new chunk. Sentences stay intact, and headings stay with the paragraphs that follow them.
- A paragraph longer than the limit gets split by words as a fallback.
- **Overlap** here means carrying the last paragraph(s), up to 50 words, into the next chunk.

In [3]:
def chunk_page(text, size=300, overlap=50):
    # 1) paragraphs, splitting any paragraph that is longer than `size` words
    paras = []
    for p in text.split("\n"):
        w = p.split()
        if len(w) > size:
            paras += [" ".join(w[i:i + size]) for i in range(0, len(w), size)]
        elif w:
            paras.append(p)
    # 2) pack paragraphs into chunks of about `size` words, with paragraph overlap
    chunks, cur, n = [], [], 0
    for p in paras:
        pw = len(p.split())
        if cur and n + pw > size:
            chunks.append("\n".join(cur))
            keep, k = [], 0
            for q in reversed(cur):                 # carry trailing paragraphs as overlap
                qw = len(q.split())
                if k + qw > overlap:
                    break
                keep.insert(0, q)
                k += qw
            cur, n = keep, k
        cur.append(p)
        n += pw
    if cur:
        chunks.append("\n".join(cur))
    return chunks

page = max(docs.values(), key=lambda d: d["words"])          # try it on the longest page
parts = chunk_page(page["text"])
print(page["title"], "|", page["words"], "words ->", len(parts), "chunks")
for i, c in enumerate(parts[:3]):
    print(f"\n--- chunk {i}: {len(c.split())} words ---\n{c[:300]}...")

Unity Catalog securable objects reference | Databricks on AWS | 5805 words -> 24 chunks

--- chunk 0: 279 words ---
Last updated on Nov 13, 2026
This page describes all securable objects in Unity Catalog. A securable object is an object defined in Unity Catalog on which privileges can be granted to a principal (user, service principal, or group).
The Unity Catalog object hierarchyâ
Securable objects in Unity Ca...

--- chunk 1: 288 words ---
Secrets store sensitive values, such as credentials and tokens, that you can govern in Unity Catalog and use in code without exposing the value.
Flows (Beta) are pipeline queries that incrementally load and process data into a target table.
There are also many other securable objects in Unity Catalo...

--- chunk 2: 118 words ---
Recipients are objects that represent an external organization or group of users that a provider shares data with.
Clean rooms are objects that represent a secure environment for collaborating with other organizations wi

**What you should see:** the longest page split into several chunks of up to about 300 words, each starting at a paragraph boundary. Look at the start of chunk 1: its first lines should repeat the end of chunk 0 (that is the overlap).

## 2.4 Chunk every page and check the result

**Concepts**
- Each chunk keeps its page's **metadata** (URL, title, section). That is how answers will cite their sources later.
- Checking the **token length of every chunk** proves nothing gets cut off by the model.

In [4]:
chunks = []
for url, d in docs.items():
    for text in chunk_page(d["text"]):
        chunks.append({"id": len(chunks), "url": url, "title": d["title"],
                       "section": d["section"], "words": len(text.split()), "text": text})

wc = [c["words"] for c in chunks]
tokens = [n_tokens(c["title"] + "\n" + c["text"]) for c in chunks]
print(f"{len(chunks)} chunks from {len(docs)} pages")
print(f"words per chunk: median {statistics.median(wc):.0f}, min {min(wc)}, max {max(wc)}")
print(f"tokens per chunk (with title): max {max(tokens)}  (limit {emb.max_seq_length})")
print("chunks over the limit:", sum(t > emb.max_seq_length for t in tokens))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (558 > 512). Running this sequence through the model will result in indexing errors


175 chunks from 28 pages
words per chunk: median 283, min 15, max 333
tokens per chunk (with title): max 1219  (limit 512)
chunks over the limit: 20


**What you should see:** a few hundred chunks, a median around 200-300 words, and **0 chunks over the limit**. Very small chunks (under about 30 words) are usually short pages; that is fine.

## 2.5 Embed the chunks

**Concepts**
- We embed **title + chunk text**. The title adds context, for example which feature a paragraph is about.
- `normalize_embeddings=True` makes every vector length 1, so the dot product equals cosine similarity
- **Asymmetric search:** questions are short and passages are long. BGE models expect a special instruction in front of the **question only**: `"Represent this sentence for searching relevant passages: "`. Leaving it out still works, just slightly worse. Model-specific details like this are worth reading in a model's card on HuggingFace.
- Batching (64 chunks at a time) is much faster than one by one. On Colab's CPU this takes about 1-3 minutes. (Runtime > Change runtime type > T4 GPU makes it faster, but is not needed.)

In [5]:
X = emb.encode([c["title"] + "\n" + c["text"] for c in chunks],
               normalize_embeddings=True, batch_size=64, show_progress_bar=True).astype("float32")
print("Embedding matrix:", X.shape, "| vector lengths about", np.round(np.linalg.norm(X, axis=1)[:3], 3))

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embedding matrix: (175, 384) | vector lengths about [1. 1. 1.]


## 2.6 Build the index

**Concepts**
- `IndexFlatIP` compares the question with **every** chunk using the inner (dot) product. It is exact and fast enough for thousands of chunks.
- At millions of chunks you would use an **approximate nearest neighbor (ANN)** index such as HNSW or IVF, trading a little accuracy for a lot of speed. On Databricks, **Vector Search** manages this for you and syncs from a Delta table.
- FAISS returns positions (0, 1, 2...), and because we added vectors in order, position `i` is `chunks[i]`.

In [6]:
import faiss
index = faiss.IndexFlatIP(X.shape[1])
index.add(X)
print("Vectors in index:", index.ntotal)

QPREFIX = "Represent this sentence for searching relevant passages: "

def dense(q, k=5):
    qv = emb.encode([QPREFIX + q], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(qv, k)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]

Vectors in index: 175


## 2.7 Try some searches

Look at **which pages** come back, not only the scores. Scores are relative

In [7]:
tests = ["What is the difference between managed and external tables?",
         "How do I control who can access a table?",
         "How can I monitor and limit usage of AI models?",
         "How do I find and browse data assets in the UI?"]
for q in tests:
    print("\nQ:", q)
    for i, s in dense(q, 3):
        print(f"  {s:.3f}  [{chunks[i]['section']}] {chunks[i]['title']}")


Q: What is the difference between managed and external tables?
  0.786  [unity-catalog] Unity Catalog best practices | Databricks on AWS
  0.785  [unity-catalog] Managed versus external assets in Unity Catalog | Databricks on AWS
  0.746  [unity-catalog] Managed versus external assets in Unity Catalog | Databricks on AWS

Q: How do I control who can access a table?
  0.746  [unity-catalog] Access control in Unity Catalog | Databricks on AWS
  0.708  [unity-catalog] Access control in Unity Catalog | Databricks on AWS
  0.708  [unity-catalog] Unity Catalog securable objects reference | Databricks on AWS

Q: How can I monitor and limit usage of AI models?
  0.741  [ai-gateway] Track model usage | Databricks on AWS
  0.732  [ai-gateway] Unity Gateway observability | Databricks on AWS
  0.723  [ai-gateway] Track model usage | Databricks on AWS

Q: How do I find and browse data assets in the UI?
  0.740  [catalog-explorer] What is Catalog Explorer? | Databricks on AWS
  0.685  [unity-catalo

**What you should see:** mostly sensible pages, for example a managed-versus-external page for the first question, access-control pages for the second, AI Gateway pages for the third and Catalog Explorer for the fourth. **Some misses are normal.** Note any query that returns the wrong pages: those become your before-and-after examples when you add hybrid search and a reranker in Steps 6-7.

## 2.8 Save everything for the next steps

**Concepts:** embedding is the slow part, so we save the results and never recompute them unless chunks change. In production, you store the model name with the vectors: vectors from different models are not comparable, so a model change means re-embedding everything.

In [8]:
json.dump(chunks, open(f"{DATA}/chunks.json", "w"))
np.save(f"{DATA}/embeddings.npy", X)
faiss.write_index(index, f"{DATA}/index.faiss")
json.dump({"embedding_model": "BAAI/bge-small-en-v1.5", "chunk_words": 300, "overlap_words": 50,
           "n_chunks": len(chunks), "dim": int(X.shape[1])},
          open(f"{DATA}/index_meta.json", "w"), indent=2)

# reload check
idx2 = faiss.read_index(f"{DATA}/index.faiss")
print("Saved and reloaded:", idx2.ntotal, "vectors |", len(json.load(open(f"{DATA}/chunks.json"))), "chunks")

Saved and reloaded: 175 vectors | 175 chunks


## Checkpoint

You are done with Step 2 when you have `chunks.json`, `embeddings.npy`, `index.faiss` and `index_meta.json` in your Drive data folder, and the test searches return mostly sensible pages. Note down the number of chunks.

**Next: Step 3, generate answers with citations.**